In [1]:
# OpenCV library import karta hai (image processing ke liye)
import cv2

# NumPy library import karta hai (arrays aur numerical operations ke liye)
import numpy as np

# Pandas library import karta hai (CSV aur tabular data handle karne ke liye)
import pandas as pd

# File aur folder operations ke liye os module import karta hai
import os

# PIL library se Image module import karta hai (OCR ke liye image open karne me kaam aata hai)
from PIL import Image

# Pytesseract import karta hai (image se text detect karne ke liye)
import pytesseract

# Tesseract executable ka path set karta hai (Windows ke liye zaroori hai)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Naye dataset folder ka path define karta hai
DATASET_DIR = r'C:\Projects\Internship_Learning\dataset_final'

# Images folder ka path banata hai
IMAGES_DIR = os.path.join(DATASET_DIR, 'images')

# labels.csv ka path define karta hai
LABELS_CSV = os.path.join(DATASET_DIR, 'labels.csv')

# labels.csv ko DataFrame me load karta hai
labels_df = pd.read_csv(LABELS_CSV)

# Filenames ki list labels.csv ke order me leta hai (os.listdir ya sorted order use NAHI karta)
filenames = labels_df['filename'].tolist()

# Total images ki sankhya print karta hai
print("Images to process:", len(filenames))

# Tesseract sahi se install/accessible hai ya nahi, version print karke check karta hai
print("Tesseract version:", pytesseract.get_tesseract_version())

# Confirm karta hai ki sabhi libraries successfully load ho gayi hain
print("All libraries loaded!")

Images to process: 979
Tesseract version: 5.5.0.20241111
All libraries loaded!


In [2]:
# ============================================================
# 02_feature_extraction_v2 - Cell 2
# extract_features(): ek image se 10 visual features nikalne ka function
# ============================================================

# Ye function image ka path leta hai aur 10 visual features ki dictionary return karta hai
def extract_features(image_path):

    # Image ko BGR format me disk se padhta hai
    img = cv2.imread(image_path)

    # Agar image padh nahi payi (corrupt/missing) to None return karta hai taaki loop crash na ho
    if img is None:
        return None

    # Saari images ko same size (1900x968) pe laata hai, kyunki old images me 1-2 px ka farak hai
    img = cv2.resize(img, (1900, 968))

    # Image ko grayscale me convert karta hai (brightness/contrast/edges ke liye)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Image ko HSV me convert karta hai (saturation nikalne ke liye)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Total pixels ki count nikalta hai (ratios ke liye)
    total_pixels = gray.shape[0] * gray.shape[1]

    # 1. brightness: grayscale pixels ka average (0-255), image kitni bright hai
    brightness = float(np.mean(gray))

    # 2. contrast: grayscale pixels ka standard deviation, zyada matlab zyada contrast
    contrast = float(np.std(gray))

    # Canny edge detector se edges nikalta hai (thresholds 100 aur 200)
    edges = cv2.Canny(gray, 100, 200)

    # 3. edge_density: edge pixels ka fraction (0-1), zyada matlab busy/detailed page
    edge_density = float(np.count_nonzero(edges) / total_pixels)

    # 4. dominant_color_count: K-Means (K=5) se dominant colors ginta hai
    # Speed ke liye image ko chhota (160x80) karta hai
    small = cv2.resize(img, (160, 80))

    # Pixels ko (N, 3) shape me float32 me reshape karta hai (cv2.kmeans ki requirement)
    pixels = small.reshape(-1, 3).astype(np.float32)

    # K-Means ke stopping criteria: 10 iterations ya 1.0 accuracy
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)

    # K-Means chalata hai, labels_km me har pixel ka cluster number milta hai
    _, labels_km, _ = cv2.kmeans(pixels, 5, None, criteria, 3, cv2.KMEANS_PP_CENTERS)

    # Har cluster me kitne pixels hain wo ginta hai
    counts = np.bincount(labels_km.flatten(), minlength=5)

    # Sirf wahi clusters ginta hai jinka share 5% se zyada ho (chhote noise clusters ignore)
    dominant_color_count = int(np.sum(counts / counts.sum() > 0.05))

    # 5. whitespace_ratio: bahut light pixels (gray > 240) ka fraction, khaali jagah ka estimate
    whitespace_ratio = float(np.count_nonzero(gray > 240) / total_pixels)

    # 6. sharpness: Laplacian ka variance, zyada matlab sharp/detailed
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    # 7. saturation_mean: HSV ke S channel ka average, colorfulness ka measure
    saturation_mean = float(np.mean(hsv[:, :, 1]))

    # 8. dark_area_ratio: dark pixels (gray < 50) ka fraction
    dark_area_ratio = float(np.count_nonzero(gray < 50) / total_pixels)

    # 9. light_area_ratio: light pixels (gray > 200) ka fraction
    light_area_ratio = float(np.count_nonzero(gray > 200) / total_pixels)

    # 10. clutter_score: edge density ko non-khaali area se multiply karta hai
    # (jitna zyada edges aur kam whitespace, utna zyada clutter)
    clutter_score = float(edge_density * (1.0 - whitespace_ratio))

    # Saare features ko dictionary me return karta hai
    return {
        "brightness": brightness,
        "contrast": contrast,
        "edge_density": edge_density,
        "dominant_color_count": dominant_color_count,
        "whitespace_ratio": whitespace_ratio,
        "sharpness": sharpness,
        "saturation_mean": saturation_mean,
        "dark_area_ratio": dark_area_ratio,
        "light_area_ratio": light_area_ratio,
        "clutter_score": clutter_score,
    }


# ---------- Quick test: pehli 2 images pe function chala ke dekhta hai ----------
for fname in filenames[:2]:

    # Image ka full path banata hai
    path = os.path.join(IMAGES_DIR, fname)

    # Features nikalta hai
    feats = extract_features(path)

    # Filename aur features print karta hai
    print(fname)
    print(feats)
    print()

old_0001.png
{'brightness': 159.41376903001304, 'contrast': 82.26522213287502, 'edge_density': 0.06726892127011744, 'dominant_color_count': 5, 'whitespace_ratio': 0.321877990430622, 'sharpness': 1233.1808521155817, 'saturation_mean': 92.28400608960418, 'dark_area_ratio': 0.12847216180948237, 'light_area_ratio': 0.40540832970856894, 'clutter_score': 0.04561653607325632}

old_0002.png
{'brightness': 192.60868747281427, 'contrast': 85.14229597652238, 'edge_density': 0.038126359286646365, 'dominant_color_count': 5, 'whitespace_ratio': 0.4790468682035668, 'sharpness': 985.353405790061, 'saturation_mean': 75.17576772509787, 'dark_area_ratio': 0.17285069595476293, 'light_area_ratio': 0.6920084819486734, 'clutter_score': 0.019862046274374448}



In [3]:
# ============================================================
# 02_feature_extraction_v2 - Cell 3
# Saari 979 images pe visual features nikalne ka loop
# ============================================================

# Time measure karne ke liye time module import karta hai
import time

# Har image ke features (dictionary) yahan jama honge
rows = []

# Jin images ka feature extraction fail hoga unke filenames yahan aayenge
failed = []

# Loop shuru hone ka time note karta hai
start_time = time.time()

# labels.csv ke filenames pe ek-ek karke loop chalata hai (index i progress ke liye)
for i, fname in enumerate(filenames, start=1):

    # Image ka full path banata hai
    path = os.path.join(IMAGES_DIR, fname)

    # Try-except taaki ek image ki problem se poora loop na ruke
    try:
        # Is image ke features nikalta hai
        feats = extract_features(path)
    except Exception as e:
        # Koi error aaye to usko print karke feats None kar deta hai
        print("Error:", fname, e)
        feats = None

    # Agar features nahi mile (None) to failed list me daal deta hai
    if feats is None:
        failed.append(fname)
    else:
        # Warna filename ko features ke saath jodta hai taaki baad me merge ho sake
        feats["filename"] = fname

        # Row ko list me add karta hai
        rows.append(feats)

    # Har 100 images ke baad progress print karta hai
    if i % 100 == 0:
        print(f"{i}/{len(filenames)} done | {time.time() - start_time:.1f} sec")

# Loop khatam hone ke baad rows se DataFrame banata hai
visual_df = pd.DataFrame(rows)

# filename column ko sabse aage laata hai (padhne me aasan)
visual_df = visual_df[["filename"] + [c for c in visual_df.columns if c != "filename"]]

# Total time print karta hai
print(f"\nTotal time: {time.time() - start_time:.1f} sec")

# Successfully process hui images ki count (979 honi chahiye)
print("Processed:", len(visual_df))

# Failed images ki count (0 honi chahiye)
print("Failed:", len(failed), failed[:10])

# DataFrame ka shape print karta hai (979, 11 aana chahiye)
print("Shape:", visual_df.shape)

100/979 done | 6.9 sec
200/979 done | 13.3 sec
300/979 done | 19.7 sec
400/979 done | 26.7 sec
500/979 done | 33.9 sec
600/979 done | 41.2 sec
700/979 done | 48.6 sec
800/979 done | 55.8 sec
900/979 done | 63.2 sec

Total time: 69.1 sec
Processed: 979
Failed: 0 []
Shape: (979, 11)


In [4]:
# ============================================================
# 02_feature_extraction_v2 - Cell 4
# visual_features.csv save karna aur describe() se sanity check
# ============================================================

# visual_features.csv ka save path banata hai (dataset_final folder ke andar)
VISUAL_CSV = os.path.join(DATASET_DIR, "visual_features.csv")

# visual_df ko CSV me save karta hai (index=False taaki extra index column na bane)
visual_df.to_csv(VISUAL_CSV, index=False)

# Save hone ka path print karta hai
print("Saved:", VISUAL_CSV)

# Shape print karta hai (979, 11)
print("Shape:", visual_df.shape)

# Null/NaN values check karta hai (sab 0 hone chahiye)
print("\nNull values per column:")
print(visual_df.isnull().sum())

# Pandas ko saare columns ek line me dikhane ke liye width badhata hai
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# Numeric columns ka summary (count, mean, std, min, max) print karta hai
print("\nSummary statistics:")
print(visual_df.describe().round(3))

Saved: C:\Projects\Internship_Learning\dataset_final\visual_features.csv
Shape: (979, 11)

Null values per column:
filename                0
brightness              0
contrast                0
edge_density            0
dominant_color_count    0
whitespace_ratio        0
sharpness               0
saturation_mean         0
dark_area_ratio         0
light_area_ratio        0
clutter_score           0
dtype: int64

Summary statistics:
       brightness  contrast  edge_density  dominant_color_count  whitespace_ratio  sharpness  saturation_mean  dark_area_ratio  light_area_ratio  clutter_score
count     979.000   979.000       979.000               979.000           979.000    979.000          979.000          979.000           979.000        979.000
mean      162.231    66.355         0.036                 3.679             0.395   1186.706           61.321            0.199             0.496          0.022
std        62.929    20.812         0.022                 1.289             0.293    

In [5]:
# ============================================================
# 02_feature_extraction_v2 - Cell 5
# OCR (Tesseract) se text features: word_count, text_coverage_ratio,
# avg_text_x, avg_text_y  ->  text_features.csv
# ============================================================

# Time measure karne ke liye (agar pehle import ho chuka hai to koi dikkat nahi)
import time

# pytesseract ka Output type import karta hai (dictionary format me OCR data lene ke liye)
from pytesseract import Output

# Ek image se OCR text features nikalne ka function
def extract_text_features(image_path):

    # Image ko BGR me padhta hai
    img = cv2.imread(image_path)

    # Agar image padh nahi payi to None return karta hai
    if img is None:
        return None

    # Saari images ko same size (1900x968) pe laata hai
    img = cv2.resize(img, (1900, 968))

    # Image ki height aur width nikalta hai (normalize karne ke liye)
    h, w = img.shape[:2]

    # OCR ke liye BGR se RGB convert karta hai (Tesseract RGB expect karta hai)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Tesseract se har word ka text, confidence aur bounding box nikalta hai
    data = pytesseract.image_to_data(rgb, output_type=Output.DICT)

    # Valid words ke bounding boxes yahan jama honge
    boxes = []

    # Har detected item pe loop chalata hai
    for i in range(len(data["text"])):

        # Word ka text nikalta hai aur extra spaces hatata hai
        txt = str(data["text"][i]).strip()

        # Word ka confidence nikalta hai (-1 ya galat value ho to -1 maan leta hai)
        try:
            conf = float(data["conf"][i])
        except (ValueError, TypeError):
            conf = -1

        # Sirf wahi words rakhta hai jo khaali nahi hain aur confidence 30 se zyada hai (noise hatane ke liye)
        if txt != "" and conf > 30:
            # Box ka left, top, width, height nikalta hai
            x = data["left"][i]
            y = data["top"][i]
            bw = data["width"][i]
            bh = data["height"][i]

            # Box ko list me add karta hai
            boxes.append((x, y, bw, bh))

    # 1. word_count: kitne valid words mile
    word_count = len(boxes)

    # Agar koi word nahi mila to sab features 0 return karta hai
    if word_count == 0:
        return {"word_count": 0, "text_coverage_ratio": 0.0,
                "avg_text_x": 0.0, "avg_text_y": 0.0}

    # Saare boxes ka total area jodta hai
    total_box_area = sum(bw * bh for (x, y, bw, bh) in boxes)

    # 2. text_coverage_ratio: text boxes ne image ka kitna hissa gher rakha hai (max 1.0)
    text_coverage_ratio = float(min(total_box_area / (w * h), 1.0))

    # 3. avg_text_x: text ke box centers ka average x position (0-1, 0=left, 1=right)
    avg_text_x = float(np.mean([(x + bw / 2) / w for (x, y, bw, bh) in boxes]))

    # 4. avg_text_y: text ke box centers ka average y position (0-1, 0=top, 1=bottom)
    avg_text_y = float(np.mean([(y + bh / 2) / h for (x, y, bw, bh) in boxes]))

    # Saare text features dictionary me return karta hai
    return {
        "word_count": word_count,
        "text_coverage_ratio": text_coverage_ratio,
        "avg_text_x": avg_text_x,
        "avg_text_y": avg_text_y,
    }


# ---------- Saari 979 images pe OCR loop ----------

# Har image ke text features yahan jama honge
text_rows = []

# OCR fail hone wali images ki list
text_failed = []

# Loop shuru hone ka time
start_time = time.time()

# labels.csv ke filenames pe loop
for i, fname in enumerate(filenames, start=1):

    # Image ka full path
    path = os.path.join(IMAGES_DIR, fname)

    # Try-except taaki ek image ke error se loop na ruke
    try:
        tfeats = extract_text_features(path)
    except Exception as e:
        print("Error:", fname, e)
        tfeats = None

    # Agar features nahi mile to failed list me daalta hai
    if tfeats is None:
        text_failed.append(fname)
    else:
        # Filename jodta hai taaki merge ho sake
        tfeats["filename"] = fname
        text_rows.append(tfeats)

    # Har 50 images pe progress print karta hai
    if i % 50 == 0:
        print(f"{i}/{len(filenames)} done | {time.time() - start_time:.1f} sec")

# Rows se DataFrame banata hai
text_df = pd.DataFrame(text_rows)

# filename column ko sabse aage laata hai
text_df = text_df[["filename"] + [c for c in text_df.columns if c != "filename"]]

# text_features.csv ka path banata hai aur save karta hai
TEXT_CSV = os.path.join(DATASET_DIR, "text_features.csv")
text_df.to_csv(TEXT_CSV, index=False)

# Summary print karta hai
print(f"\nTotal time: {time.time() - start_time:.1f} sec")
print("Processed:", len(text_df))
print("Failed:", len(text_failed), text_failed[:10])
print("Saved:", TEXT_CSV)
print("\nSummary statistics:")
print(text_df.describe().round(3))

50/979 done | 31.3 sec
100/979 done | 57.9 sec
150/979 done | 90.1 sec
200/979 done | 123.3 sec
250/979 done | 157.5 sec
300/979 done | 190.5 sec
350/979 done | 222.4 sec
400/979 done | 254.4 sec
450/979 done | 286.5 sec
500/979 done | 320.7 sec
550/979 done | 351.3 sec
600/979 done | 385.2 sec
650/979 done | 419.2 sec
700/979 done | 611.2 sec
750/979 done | 653.0 sec
800/979 done | 690.4 sec
850/979 done | 730.8 sec
900/979 done | 772.8 sec
950/979 done | 814.6 sec

Total time: 833.4 sec
Processed: 979
Failed: 0 []
Saved: C:\Projects\Internship_Learning\dataset_final\text_features.csv

Summary statistics:
       word_count  text_coverage_ratio  avg_text_x  avg_text_y
count     979.000              979.000     979.000     979.000
mean       72.382                0.046       0.462       0.453
std        57.598                0.034       0.112       0.172
min         0.000                0.000       0.000       0.000
25%        33.000                0.023       0.412       0.356
50%     

In [6]:
# ============================================================
# 02_feature_extraction_v2 - Cell 6
# visual + text + labels ko merge karke dataset_final\features.csv banana
# ============================================================

# labels.csv se sirf kaam ke columns leta hai: filename, label, score, source
meta_df = labels_df[["filename", "label", "score", "source"]]

# Visual features ko text features ke saath filename pe merge karta hai
# (how="inner" taaki sirf dono me maujood filenames aayein)
features_df = visual_df.merge(text_df, on="filename", how="inner")

# Ab labels/score/source ko bhi filename pe jodta hai
features_df = features_df.merge(meta_df, on="filename", how="inner")

# Rows ki count check karta hai: 979 hi honi chahiye, warna AssertionError aayega
assert len(features_df) == 979, f"Rows mismatch: {len(features_df)}"

# Duplicate filenames nahi hone chahiye
assert features_df["filename"].is_unique, "Duplicate filenames mile!"

# features.csv ka path banata hai aur save karta hai
FEATURES_CSV = os.path.join(DATASET_DIR, "features.csv")
features_df.to_csv(FEATURES_CSV, index=False)

# Save confirmation aur shape print karta hai (979, 19 aana chahiye)
print("Saved:", FEATURES_CSV)
print("Shape:", features_df.shape)
print("Columns:", list(features_df.columns))

# Pandas display settings (saare columns ek line me dikhane ke liye)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# Feature columns ki list (filename, label, score, source ko chhod ke)
feature_cols = [c for c in features_df.columns if c not in ["filename", "label", "score", "source"]]

# Label-wise mean print karta hai (good/average/poor me features kaise alag hain)
print("\nLabel-wise mean:")
print(features_df.groupby("label")[feature_cols].mean().round(3).T)

# Source-wise mean print karta hai (old/b1/b2/b3 me bada farak ho to source

Saved: C:\Projects\Internship_Learning\dataset_final\features.csv
Shape: (979, 18)
Columns: ['filename', 'brightness', 'contrast', 'edge_density', 'dominant_color_count', 'whitespace_ratio', 'sharpness', 'saturation_mean', 'dark_area_ratio', 'light_area_ratio', 'clutter_score', 'word_count', 'text_coverage_ratio', 'avg_text_x', 'avg_text_y', 'label', 'score', 'source']

Label-wise mean:
label                  average     good      poor
brightness             169.505  146.275   170.587
contrast                68.615   60.928    69.513
edge_density             0.035    0.031     0.042
dominant_color_count     3.674    3.715     3.644
whitespace_ratio         0.431    0.325     0.426
sharpness             1172.910  901.096  1548.247
saturation_mean         58.979   63.873    61.717
dark_area_ratio          0.180    0.256     0.159
light_area_ratio         0.533    0.416     0.536
clutter_score            0.021    0.022     0.025
word_count              76.397   50.339    92.809
text_cover

In [7]:
# ============================================================
# 02_feature_extraction_v2 - Cell 6b
# Source-wise mean aur label x source crosstab dobara print karta hai
# ============================================================

# Display settings: saare columns ek line me dikhane ke liye
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# Feature columns ki list (filename, label, score, source chhod ke)
feature_cols = [c for c in features_df.columns if c not in ["filename", "label", "score", "source"]]

# Source-wise mean: old/b1/b2/b3 me features ka farak (source bias ka sanket)
print("Source-wise mean:")
print(features_df.groupby("source")[feature_cols].mean().round(3).T)

# Label x source counts: kis source me kaunsi class kitni hai
print("\nLabel x Source counts:")
print(pd.crosstab(features_df["source"], features_df["label"]))

Source-wise mean:
source                      b1        b2        b3      old
brightness             154.360   164.035   154.784  177.179
contrast                66.695    65.237    66.846   67.036
edge_density             0.035     0.034     0.036    0.039
dominant_color_count     3.705     3.606     3.724    3.700
whitespace_ratio         0.384     0.408     0.356    0.449
sharpness             1341.505  1253.198  1225.886  915.457
saturation_mean         68.661    60.055    62.784   55.972
dark_area_ratio          0.230     0.196     0.214    0.157
light_area_ratio         0.458     0.508     0.453    0.575
clutter_score            0.022     0.021     0.024    0.022
word_count              65.129    72.074    65.620   89.040
text_coverage_ratio      0.038     0.046     0.040    0.061
avg_text_x               0.461     0.463     0.456    0.473
avg_text_y               0.442     0.455     0.437    0.482

Label x Source counts:
label   average  good  poor
source                     
b1